# Gemini Batch Jobs — Cost, Time & Token Analysis

Fetches all batch jobs from the Gemini API and breaks down:
- **Job status & timing** (duration, state)
- **Token usage** (prompt / output / thoughts per job and per request)
- **Estimated cost** using Gemini pricing (batch = 50% discount vs. standard)
- **Per-request breakdowns** (error rate, token distribution)

In [ ]:
import os
import sys
import math
import json
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from datetime import datetime, timezone
from IPython.display import display

from google import genai
from google.genai import types

API_KEY = os.getenv("GOOGLE_AI_WIKI_KEY")
assert API_KEY, "Set the GOOGLE_AI_WIKI_KEY environment variable before running."

client = genai.Client(api_key=API_KEY)
print("Client ready.")

## 1 · Pricing table

Edit the prices (USD per **1 M tokens**) to match the current Gemini pricing page.  
Batch API requests are billed at **50 % of the standard price**.

In [ ]:
# Prices in USD per 1 M tokens (standard / non-batch)
# Source: https://ai.google.dev/gemini-api/docs/pricing  (update as needed)
PRICING = {
    # model-name-fragment -> {input, output}  (all in $/1M tokens)
    "gemini-4-carbon": {"input": 2.00, "output": 10.00},
    "gemini-4-neon": {"input": 3.00, "output": 15},
    "gemini-4-hydrogen": {"input": 1.50, "output": 7.50},
    "gemini-3.8-flash":  {"input": 0.75, "output": 3.75},
}

BATCH_DISCOUNT = 0.50   # batch API = 50 % of standard price

def get_price(model_name: str) -> dict:
    model_lower = (model_name or "").lower().split("/")[-1]
    return PRICING[model_lower]

def estimate_cost(input_tokens: int, output_tokens: int, model_name: str, batch: bool = True) -> float:
    prices = get_price(model_name)
    cost = (input_tokens * prices["input"] + output_tokens * prices["output"]) / 1_000_000
    if batch:
        cost *= BATCH_DISCOUNT
    return cost

print("Pricing table loaded.")

## 2 · Fetch all batch jobs

In [ ]:
ark_id_dicts = {}
SOURCES = ['equipe', 'figaro', 'franc_tireur', 'humanite', 'liberation']

for source in SOURCES :
    with open(f"gallica_analysis/cache/ark_ids/{source}_ark_ids.json", "r", encoding="utf-8") as f:
        d = json.load(f)
        for key in d.values():
            ark_id_dicts[key] = source

In [ ]:
all_batches = list(client.batches.list())
print(f"Found {len(all_batches)} batch job(s).")

for b in all_batches:
    batch_job = client.batches.get(name=b.name)
    ark_id = batch_job.dest.inlined_responses[0].metadata.get("ark_id")
    print(f"  {b.name}  |  {b.display_name}  |  state={b.state}  |  model={b.model} | ark_id={ark_id} | source={ark_id_dicts[ark_id]}")

## 3 · Build the jobs summary DataFrame

In [ ]:
def duration_seconds(start, end) -> float | None:
    """Return (end - start) in seconds, handling timezone-naive vs aware."""
    if start is None or end is None:
        return None
    # make both tz-aware (UTC) if they aren't already
    if start.tzinfo is None:
        start = start.replace(tzinfo=timezone.utc)
    if end.tzinfo is None:
        end = end.replace(tzinfo=timezone.utc)
    return (end - start).total_seconds()


rows = []
for b in all_batches:
    # ------- timing -------
    dur = duration_seconds(b.start_time or b.create_time, b.end_time or b.update_time)

    # ------- token totals (summed across all inlined responses) -------
    prompt_tokens = 0
    output_tokens = 0
    thoughts_tokens = 0
    total_tokens = 0
    n_requests = 0
    n_errors = 0

    batch_job = client.batches.get(name=b.name)

    if batch_job.dest and batch_job.dest.inlined_responses:
        for resp in batch_job.dest.inlined_responses:
            n_requests += 1
            if resp.error is not None:
                n_errors += 1
                continue
            if resp.response and resp.response.usage_metadata:
                u = resp.response.usage_metadata
                prompt_tokens   += u.prompt_token_count    or 0
                output_tokens   += u.candidates_token_count or 0
                thoughts_tokens += u.thoughts_token_count  or 0
                total_tokens    += u.total_token_count     or 0

    cost_usd = estimate_cost(prompt_tokens, output_tokens + thoughts_tokens, b.model or "", batch=True)
    
    ark_id = batch_job.dest.inlined_responses[0].metadata.get("ark_id")
    rows.append({
        "name":            b.name,
        "display_name":    b.display_name,
        "model":           b.model,
        "state":           str(b.state),
        "source":          ark_id_dicts[ark_id],
        "create_time":     b.create_time,
        "end_time":        b.end_time,
        "duration_min":    round(dur / 60, 2) if dur is not None else None,
        "n_requests":      n_requests,
        "n_errors":        n_errors,
        "prompt_tokens":   prompt_tokens,
        "output_tokens":   output_tokens,
        "thoughts_tokens": thoughts_tokens,
        "total_tokens":    total_tokens,
        "cost_usd":        round(cost_usd, 6),
    })

jobs_df = pd.DataFrame(rows)
display(jobs_df)

In [ ]:
display(jobs_df[['source', 'prompt_tokens', 'output_tokens', 'thoughts_tokens', 'cost_usd']])

## 4 · Summary statistics

In [ ]:
import pandas as pd

numeric_cols = [
    "duration_min", "n_requests", "n_errors",
    "prompt_tokens", "output_tokens", "thoughts_tokens",
    "total_tokens", "cost_usd"
]

with pd.option_context("display.float_format", lambda x: f"{x:,.6f}".rstrip("0").rstrip(".")):
    print("=== Totals across all jobs ===")
    totals = jobs_df[numeric_cols].sum(numeric_only=True)
    print(totals)

    print("\n=== Per-job summary ===")
    display(jobs_df[numeric_cols].describe().round(3))